# ARC-AGI-3 · S10: корректная память поверх S4

**Готовый кандидат для Kaggle и понятная база для продолжения.** Дата: 22 сентября 2026.

Агент получает игровое поле, рассуждает локальной Qwen3.8 Flash Next NVFP4,
пишет Python для анализа и выбирает действия. Duck/TAAF исполняет их и возвращает
новое наблюдение. S10 исправляет привязку памяти бесполезных действий к игре и
конкретному переходу. Новая память даёт совет модели, не запрещает действия.

**Что уже измерено:** родитель S4 получил Public LB **3.70**, точный повтор — **3.38**.
Среднее двух запусков 3.54 — описание этих наблюдений, не прогноз нового результата.
**У S10 пока нет измеренного Public/Private LB.** Проверки кода и короткий smoke
не доказывают рост качества. Сохранён профиль `s4_baseline` для честного сравнения.

## Карта ноутбука

1. Настройки и режим запуска → 2. Среда и модель → 3. Читаемые модули памяти →
4. Установка выбранного профиля → 5. Игры и запуск → 6. Результаты → 7. Развитие.

Обозначения: **LB** — таблица конкурса; **smoke** — короткая проверка работоспособности;
**panel** — фиксированный набор публичных игр; **gateway** — сервис скрытых игр Kaggle.
Весь новый код ниже виден обычным текстом; локальные файлы проекта для запуска не нужны.

## 1. Быстрый старт и выбор режима

В Kaggle нужен **NVIDIA RTX PRO 6000**, Internet **Off** и следующие Inputs:

| Тип | Вход |
|---|---|
| Competition | `arc-prize-2026-arc-agi-3` |
| Dataset | `keithtyser/duck-qwen38-nvfp4-mtp-vllm-smoke-v1` |
| Dataset | `keithtyser/qwen38-flash-next-vllm-nvfp4-runtime-v1` |
| Model | `keithtyser/qwen3-8-flash-next-nvfp4/PyTorch/radixark-modelopt-fp4/1` |

Приложенный `kernel-metadata.json` содержит эти входы и проверенный образ Docker.
Все вычисления модели выполняются внутри Kaggle, внешнего API нет.

**Save & Run All** с настройками ниже проверяет 2 игры × максимум 4 действия.
Это Phase A. Для официального балла затем нужна отдельная **Submit to Competition**
с `submission.parquet`: в Phase B ноутбук сам переключается на все игры gateway,
полный бюджет и стохастическую генерацию. Малые лимиты smoke туда не попадают.
Полный запуск может занять до 9 часов; час разработки не является лимитом оценки.

После смены профиля перезапустите сессию и выполните все ячейки по порядку.

In [ ]:
# Единственная ячейка повседневных настроек.
PROFILE = "s10_scoped"          # "s4_baseline" — точный алгоритм контрольного S4
OFFLINE_MODE = "smoke"          # "smoke" | "panel" | "public25"
EVAL_PANEL = "validation"       # development | validation | held_public
EVAL_SEED = -1                  # -1: исходная стохастика; 101/202/303: offline A/B
PANEL_SECONDS = 600.0           # Только panel: лимит одной игры
PANEL_ACTIONS = 80              # Только panel: лимит действий одной игры
PANEL_CONCURRENCY = 4           # Только panel: одновременно запущенные игры


### Почему эта основа

| Вариант | Собственный Public LB | Решение |
|---|---:|---|
| Flash v3 | 3.39 / 2.95 | исходный serving и агент |
| S4: память no-impact, исключены 2 верхние строки HUD | 3.70 / 3.38 | основа |
| Другой открытый Duck Anim, собственный повтор | 2.92 | ниже S4 |
| Сокращение истории / новый scheduler | 2.82 / 2.82 | не добавлены |
| Анимации / ACTION7 / RESET guard | 3.13 / 3.10 / 2.81 | не добавлены |
| S10: изоляция и корректность памяти | ещё не измерен | кандидат этого ноутбука |

Результаты из истории собственных отправок, проверенной 22.09.2026. Это не сравнение
с лидерами конкурса. Размер выборки слишком мал для надёжной оценки дисперсии.
У свежего открытого «Tuned» найдены лишь изменения таймаутов, а не новый алгоритм.

## 2. Офлайн-среда и проверенный профиль модели

Ниже сохранены профиль vLLM, NVFP4, MTP=3 и настройки исходного S4.
MTP — ускорение генерации несколькими предложенными токенами; это часть serving,
а не дополнительный поиск решений. Веса не обучаются в этом ноутбуке.
Начало общего 9-часового бюджета фиксируется до установки зависимостей.

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# Skip periodic JSON/HTML diagnostics and per-frame logging for every run.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1"

# Apply the measured vLLM winner before any serving setup command runs.
PUBLIC25_VLLM_PROFILE_NAME = 'kv5-bf16-mtp3-c8-cg32'
PUBLIC25_VLLM_PROFILE_ENV = {
    "TAAF_VLLM_ENABLE_PREFIX_CACHING": "0",
    "TAAF_VLLM_KV_CACHE_DTYPE": "auto",
    "TAAF_VLLM_KV_CACHE_MEMORY_BYTES": "5368709120",
    "TAAF_VLLM_MAX_CUDAGRAPH_CAPTURE_SIZE": "32",
    "TAAF_VLLM_MAX_NUM_BATCHED_TOKENS": "8192",
    "TAAF_VLLM_MAX_NUM_SEQS": "8",
    "TAAF_VLLM_MTP_TOKENS": "3",
    "TAAF_VLLM_OMP_THREADS": "1"
}
for key, value in PUBLIC25_VLLM_PROFILE_ENV.items():
    os.environ[key] = value
print(
    f'PUBLIC25_VLLM_PROFILE name={PUBLIC25_VLLM_PROFILE_NAME} '
    f'env={json.dumps(PUBLIC25_VLLM_PROFILE_ENV, sort_keys=True)}',
    flush=True,
)
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

def _competition_root():
    candidates = [
        Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3"),
        Path("/kaggle/input/arc-prize-2026-arc-agi-3"),
    ]
    for root in candidates:
        if (root / "arc_agi_3_wheels").exists():
            print(f"taaf.kaggle: competition root = {root}")
            return root
    for wheels in Path("/kaggle/input").rglob("arc_agi_3_wheels"):
        if wheels.is_dir():
            print(f"taaf.kaggle: competition root = {wheels.parent}")
            return wheels.parent
    raise RuntimeError("Competition dataset with arc_agi_3_wheels was not mounted under /kaggle/input")

COMPETITION_ROOT = _competition_root()


In [ ]:
import hashlib
import importlib
import platform

if PROFILE not in {"s10_scoped", "s4_baseline"}:
    raise ValueError("Unknown PROFILE")
if not TRUE_SUBMISSION and OFFLINE_MODE not in {"smoke", "panel", "public25"}:
    raise ValueError("Unknown OFFLINE_MODE")
print({"profile": PROFILE, "mode": "competition" if TRUE_SUBMISSION else OFFLINE_MODE,
       "python": platform.python_version()})


### 2.1. Пакеты из competition wheelhouse

Установка использует только приложенные колёса. Ошибка здесь обычно означает
неподключённый Competition Input или неподходящий Kaggle image.

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        str(COMPETITION_ROOT / "arc_agi_3_wheels"),
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)


### 2.2. Подключение исходного Duck/TAAF и запуск vLLM

Bundle содержит исходники runner и сохранённую конфигурацию. Ноутбук проверяет,
что bundle единственный; при другом наборе Inputs лучше остановиться, чем молча
выбрать случайный код. Setup запускает локальный сервер модели.

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["keithtyser/duck-qwen38-nvfp4-mtp-vllm-smoke-v1", "keithtyser/qwen38-flash-next-vllm-nvfp4-runtime-v1"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    markers = sorted(Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER))
    if len(markers) != 1:
        raise RuntimeError(f"Expected one TAAF bundle, found {len(markers)}: {markers}")
    return markers[0].parent


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")


In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)


In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR


## 3. Модули: что меняется в S10

Один объект памяти на весь процесс S4 мог смешивать параллельные игры. Также
Python-вызов без действия повторно использовал старую сводку, а серия LEFT→RIGHT,
вернувшая поле назад, могла объявить RIGHT бесполезным.

S10 связывает память с агентом, путём состояния и уровнем; принимает только свежий
переход с одним реально исполненным действием; учитывает координаты клика;
сбрасывает факты после RESET/смены уровня и ограничивает размер памяти.
Подсказка выводится только для текущего состояния и не загрязняет знания модели.

Сохранена исходная гипотеза S4: две верхние строки считаются HUD. Это остаётся
ограничением для незнакомых игр; отдельная абляция необходима, прежде чем расширять
маску или автоматически определять интерфейс. Цветам не назначены универсальные
значения, готовых ответов и правил по game ID в агенте нет.

Ячейки `%%writefile` сохраняют **показанный** код в `/kaggle/working`.
Исходники S10 можно редактировать здесь или в `src/s10_scoped_no_impact.py` проекта.

### 3.1. Режимы и защита полного бюджета Phase B

In [ ]:
%%writefile /kaggle/working/arc3_runtime.py
"""Small, portable runtime contract for the readable ARC3 S10 notebook.

The competition branch always wins over user-facing offline settings. Nothing
here chooses game actions, game order, model sampling, or hidden game IDs.
"""
from __future__ import annotations

import hashlib
import json
from pathlib import Path
from typing import Any

PROFILES = ("s10_scoped", "s4_baseline")
OFFLINE_MODES = ("smoke", "panel", "public25")
PRODUCTION = {
    "max_runtime_s_per_game": 7920.0,
    "analyzer_timeout": 900.0,
    "concurrency": 28,
    "max_actions_per_game": None,
}
SMOKE = {
    "max_runtime_s_per_game": 180.0,
    "analyzer_timeout": 120.0,
    "concurrency": 2,
    "max_actions_per_game": 4,
}


def resolve_run_config(*, true_submission: bool, profile: str,
                       offline_mode: str = "smoke", panel: str = "validation",
                       eval_seed: int = -1, panel_seconds: float = 600.0,
                       panel_actions: int = 80, panel_concurrency: int = 4) -> dict:
    if profile not in PROFILES:
        raise ValueError(f"Unknown profile {profile!r}; choose {PROFILES}")
    if true_submission:
        # Offline selection, seeds and tiny smoke caps cannot affect Phase B.
        return {"profile": profile, "mode": "competition", "panel": None,
                "seed": -1, "budgets": dict(PRODUCTION)}
    if offline_mode not in OFFLINE_MODES:
        raise ValueError(f"Unknown offline mode {offline_mode!r}")
    if not isinstance(eval_seed, int) or eval_seed < -1:
        raise ValueError("eval_seed must be -1 (stochastic) or a non-negative int")
    if panel not in {"development", "validation", "held_public"}:
        raise ValueError("Unknown evaluation panel")
    budgets = dict(PRODUCTION if offline_mode == "public25" else SMOKE)
    if offline_mode == "panel":
        if not 30 <= panel_seconds <= 7920 or not 1 <= panel_actions <= 10000:
            raise ValueError("Invalid panel runtime/action cap")
        if not 1 <= panel_concurrency <= 28:
            raise ValueError("panel_concurrency must be between 1 and 28")
        budgets = {"max_runtime_s_per_game": float(panel_seconds),
                   "analyzer_timeout": min(900.0, float(panel_seconds)),
                   "concurrency": int(panel_concurrency),
                   "max_actions_per_game": int(panel_actions)}
    return {"profile": profile, "mode": offline_mode,
            "panel": panel if offline_mode == "panel" else None,
            "seed": eval_seed, "budgets": budgets}


def apply_run_config(bm: Any, target: Any, config: dict) -> None:
    if float(getattr(target, "max_runtime_s", 0) or 0) != 32400.0:
        raise RuntimeError("Source bundle must declare the verified 9-hour budget")
    for key, value in config["budgets"].items():
        if not hasattr(bm.solver, key):
            raise RuntimeError(f"Solver contract changed: missing {key}")
        setattr(bm.solver, key, value)
    bm.solver.save_request_logs = False
    bm.n_passes = 1
    bm.game_weights = None


def select_offline_ids(public_ids: list[str], panels: dict, config: dict) -> list[str]:
    if config["mode"] == "competition":
        raise RuntimeError("Competition must enumerate the live gateway, never public IDs")
    if len(public_ids) != 25 or len(set(public_ids)) != 25:
        raise RuntimeError("Expected 25 distinct pinned public games")
    split = [g for name in ("development", "validation", "held_public") for g in panels[name]]
    if len(split) != 25 or set(split) != set(public_ids):
        raise RuntimeError("Evaluation panels must partition all public games")
    if config["mode"] == "smoke":
        return public_ids[:2]
    if config["mode"] == "panel":
        return list(panels[config["panel"]])
    return list(public_ids)


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def source_fingerprints(paths: dict[str, Path]) -> dict[str, str]:
    """Hash the small executable inputs; never scan model weights or hidden files."""
    return {label: file_sha256(path) for label, path in sorted(paths.items())}


def write_manifest(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str) + "\n",
                    encoding="utf-8")


### 3.2. Контрольный S4 — сохранён без алгоритмических изменений

In [ ]:
%%writefile /kaggle/working/arc3_s4_baseline.py
"""S4 semantic no-impact guard: HUD-insensitive repeats, not a global ban.

Single causal change: compare frames after stripping the top HUD strip, remember
``(semantic_state, action)`` only after a confirmed no-interior-change repeat,
and tell the model. Real interior motion and animations are not banned.
"""

from __future__ import annotations

import hashlib
import json
import threading
import time
from pathlib import Path
from typing import Any, Callable

S4_EXPERIMENT_ID = "S4"
S4_TELEMETRY_NAME = "s4_telemetry.json"
S4_PATCH_FLAG = "_arc_s4_patched"
HUD_TOP_ROWS = 2
CONFIRM_REPEATS = 2

Grid = list[list[int]]

_LOCK = threading.Lock()
_COUNTERS = {
    "transitions": 0,
    "identical": 0,
    "hud_only": 0,
    "real_change": 0,
    "confirmed_no_impact": 0,
}


def parse_grid(value: Any) -> Grid | None:
    if value is None:
        return None
    if isinstance(value, dict) and "grid" in value:
        value = value["grid"]
    rows: Grid = []
    if isinstance(value, str):
        for line in value.splitlines():
            stripped = line.strip()
            if not stripped:
                continue
            rows.append([int(ch) if ch.isdigit() else ord(ch) % 16 for ch in stripped])
    else:
        try:
            for row in value:
                rows.append([int(cell) for cell in row])
        except (TypeError, ValueError):
            return None
    if not rows or not rows[0]:
        return None
    width = len(rows[0])
    if any(len(row) != width for row in rows):
        return None
    return rows


def strip_hud(grid: Grid, *, top_rows: int = HUD_TOP_ROWS) -> Grid:
    if len(grid) <= top_rows:
        return [list(row) for row in grid]
    return [list(row) for row in grid[top_rows:]]


def semantic_key(grid: Grid, *, top_rows: int = HUD_TOP_ROWS) -> str:
    interior = strip_hud(grid, top_rows=top_rows)
    payload = json.dumps(interior, separators=(",", ":"), ensure_ascii=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def classify_transition(
    before: Grid,
    after: Grid,
    *,
    top_rows: int = HUD_TOP_ROWS,
) -> str:
    if before == after:
        return "identical"
    if semantic_key(before, top_rows=top_rows) == semantic_key(after, top_rows=top_rows):
        return "hud_only"
    return "real_change"


def action_key(action: Any) -> str:
    if isinstance(action, dict):
        name = str(action.get("action") or action.get("action_display") or "").strip()
        row = action.get("row")
        col = action.get("col")
        if row is not None or col is not None:
            return f"{name}:{row}:{col}"
        return name
    return str(action or "").strip()


class NoImpactBook:
    """Per-game memory. Layout-specific; caller must reset on level change."""

    def __init__(self, *, confirm_repeats: int = CONFIRM_REPEATS) -> None:
        self.confirm_repeats = confirm_repeats
        self._counts: dict[tuple[str, str], int] = {}
        self._confirmed: set[tuple[str, str]] = set()

    def reset(self) -> None:
        self._counts.clear()
        self._confirmed.clear()

    def observe(self, state_key: str, action: str, kind: str) -> bool:
        pair = (state_key, action)
        if kind == "real_change":
            self._counts.pop(pair, None)
            self._confirmed.discard(pair)
            return False
        if kind not in {"identical", "hud_only"}:
            return False
        self._counts[pair] = self._counts.get(pair, 0) + 1
        if self._counts[pair] >= self.confirm_repeats:
            self._confirmed.add(pair)
            return True
        return False

    def is_no_impact(self, state_key: str, action: str) -> bool:
        return (state_key, action) in self._confirmed

    def notes_for(self, state_key: str) -> list[str]:
        notes = []
        for stored_state, action in sorted(self._confirmed):
            if stored_state == state_key:
                notes.append(
                    f"{action} is confirmed no-impact in this semantic state "
                    "(HUD/timer-only or identical interior); try a different action."
                )
        return notes


def format_no_impact_note(notes: list[str]) -> str:
    if not notes:
        return ""
    return "No-impact memory: " + " ".join(notes)


def inject_no_impact_notes(knowledge: dict[str, str], notes: list[str]) -> None:
    blob = format_no_impact_note(notes)
    if not blob:
        return
    current = str(knowledge.get("recent_findings") or "").strip()
    if blob in current:
        return
    knowledge["recent_findings"] = f"{blob} {current}".strip()


def wrap_run_python_tool(
    original: Callable[..., Any],
    *,
    book: NoImpactBook,
    load_grid: Callable[[Any], Grid | None],
) -> Callable[..., Any]:
    def wrapped(self: Any, state_path: Any, arguments: dict[str, Any]) -> Any:
        before = load_grid(state_path)
        result = original(self, state_path, arguments)
        after = load_grid(state_path)
        summary = getattr(self, "_last_step_summary", None) or {}
        if summary.get("level_transition") or summary.get("run_complete") or summary.get("game_over"):
            book.reset()
            return result
        action = action_key(
            (summary.get("executed_actions") or [None])[-1]
            if isinstance(summary.get("executed_actions"), list)
            else summary.get("executed_actions")
        )
        if not action:
            last = getattr(self, "_last_action_result", None) or {}
            action = action_key(last.get("action_display") or last.get("action"))
        if before is None or after is None or not action:
            return result
        kind = classify_transition(before, after)
        with _LOCK:
            _COUNTERS["transitions"] += 1
            _COUNTERS[kind] = _COUNTERS.get(kind, 0) + 1
        newly = book.observe(semantic_key(before), action, kind)
        if newly:
            with _LOCK:
                _COUNTERS["confirmed_no_impact"] += 1
            print(
                f"S4_NO_IMPACT confirmed action={action} kind={kind}",
                flush=True,
            )
        knowledge = getattr(self, "_summarized_knowledge", None)
        if isinstance(knowledge, dict):
            inject_no_impact_notes(knowledge, book.notes_for(semantic_key(after)))
        return result

    setattr(wrapped, S4_PATCH_FLAG, True)
    return wrapped


def duck_load_grid(state_path: Any) -> Grid | None:
    from inference.agent import tool_agent as ta

    loader = getattr(ta, "load_runtime_state", None)
    if loader is None:
        from inference.runtime_state import load_runtime_state as loader  # type: ignore
    frame, _history = loader(state_path)
    if frame is None:
        return None
    grid = getattr(frame, "grid", None)
    return parse_grid(grid)


def telemetry_path(working_dir: Path) -> Path:
    return Path(working_dir) / S4_TELEMETRY_NAME


def write_telemetry(working_dir: Path, payload: dict[str, Any]) -> Path | None:
    path = telemetry_path(working_dir)
    try:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(
            json.dumps(payload, indent=2, sort_keys=True, default=str) + "\n",
            encoding="utf-8",
        )
        print(f"S4 telemetry written: {path}", flush=True)
        return path
    except Exception as exc:
        print(f"S4 telemetry write failed: {exc!r}", flush=True)
        return None


def install_s4_hooks(
    bm: Any,
    *,
    target: Any | None = None,
    working_dir: Path,
    bundle_dir: Path | None = None,
    extra: dict[str, Any] | None = None,
    source_text: str | None = None,
) -> dict[str, Any]:
    from inference.agent import tool_agent as ta

    book = NoImpactBook()
    if not getattr(ta.ToolAgent._run_python_tool, S4_PATCH_FLAG, False):
        ta.ToolAgent._run_python_tool = wrap_run_python_tool(  # type: ignore[method-assign]
            ta.ToolAgent._run_python_tool,
            book=book,
            load_grid=duck_load_grid,
        )

    orig_run = bm.run
    if not getattr(orig_run, S4_PATCH_FLAG, False):

        async def wrapped_run(*args: Any, **kwargs: Any) -> Any:
            started = time.perf_counter()
            try:
                return await orig_run(*args, **kwargs)
            finally:
                payload = {}
                path = telemetry_path(Path(working_dir))
                if path.is_file():
                    try:
                        payload = json.loads(path.read_text(encoding="utf-8"))
                    except Exception:
                        payload = {}
                with _LOCK:
                    payload["counters"] = dict(_COUNTERS)
                payload["runtime_seconds"] = time.perf_counter() - started
                payload["finished_at_epoch"] = time.time()
                write_telemetry(Path(working_dir), payload)

        setattr(wrapped_run, S4_PATCH_FLAG, True)
        bm.run = wrapped_run

    payload = {
        "experiment_id": S4_EXPERIMENT_ID,
        "single_change": (
            "HUD-insensitive semantic diff and confirmed state-action no-impact memory"
        ),
        "hud_top_rows": HUD_TOP_ROWS,
        "confirm_repeats": CONFIRM_REPEATS,
        "global_ban": False,
        "s1_not_stacked": True,
        "s2_not_stacked": True,
        "s3_not_stacked": True,
        "installed_at_epoch": time.time(),
        "source_sha256": (
            hashlib.sha256(source_text.encode("utf-8")).hexdigest()
            if source_text is not None
            else None
        ),
        "working_dir": str(working_dir),
        "counters": dict(_COUNTERS),
    }
    if extra:
        payload["extra"] = extra
    if target is not None:
        payload["target_max_runtime_s"] = getattr(target, "max_runtime_s", None)
    if bundle_dir is not None:
        payload["bundle_dir"] = str(bundle_dir)
    write_telemetry(Path(working_dir), payload)
    print("S4 no-impact guard installed: HUD-insensitive confirmed repeats only", flush=True)
    return payload


### 3.3. Кандидат S10 — память с корректной областью действия

In [ ]:
%%writefile /kaggle/working/arc3_s10_memory.py
"""S10: bounded, scoped S4 advice with fresh single-primitive attribution.

This file is embedded verbatim in the submission notebook. It uses only the
stdlib until installation imports the harness. No actions are blocked and no
serving, prompt, or runtime budgets are changed.

Harness contract: _run_python_tool returns step_executed and replaces its
summary only when actions execute. Summary counts include every action command;
runtime frame.step and summary action numbers count primitive actions. Runtime
history contains coordinate-specific action displays and explicit RESET entries.
"""

from __future__ import annotations

from collections import OrderedDict
import functools
import hashlib
import inspect
import json
from pathlib import Path
import threading
import time
from typing import Any, Callable

S10_EXPERIMENT_ID = "S10"
S10_PATCH_FLAG = "_arc_s10_patched"
S10_TELEMETRY_NAME = "s10_telemetry.json"
HUD_TOP_ROWS = 2
CONFIRM_REPEATS = 2
MAX_PATHS_PER_AGENT = 8
MAX_PAIRS_PER_PATH = 512
MAX_NOTES_PER_STATE = 8
_S10_AGENT_ATTRIBUTE = "_arc_s10_memory"
_LOCK = threading.Lock()
_COUNTERS = {
    "transitions": 0, "identical": 0, "hud_only": 0, "real_change": 0,
    "confirmed_no_impact": 0, "skipped_no_action": 0,
    "skipped_multiple_actions": 0, "skipped_unverified": 0,
    "scope_resets": 0, "internal_errors": 0,
}


def _count(name: str) -> None:
    with _LOCK:
        _COUNTERS[name] += 1


def parse_grid(value: Any) -> list[list[int]] | None:
    if isinstance(value, dict):
        value = value.get("grid")
    try:
        if isinstance(value, str):
            rows = [[int(c) if c.isdigit() else ord(c) % 16 for c in line.strip()]
                    for line in value.splitlines() if line.strip()]
        else:
            rows = [[int(cell) for cell in row] for row in value]
    except (TypeError, ValueError):
        return None
    if not rows or not rows[0] or any(len(row) != len(rows[0]) for row in rows):
        return None
    return rows


def semantic_key(grid: list[list[int]]) -> str:
    interior = grid[HUD_TOP_ROWS:] if len(grid) > HUD_TOP_ROWS else grid
    return hashlib.sha256(json.dumps(interior, separators=(",", ":")).encode()).hexdigest()


def classify_transition(before: list[list[int]], after: list[list[int]]) -> str:
    if before == after:
        return "identical"
    return "hud_only" if semantic_key(before) == semantic_key(after) else "real_change"


def action_key(action: Any) -> str:
    if isinstance(action, dict):
        name = str(action.get("action") or action.get("action_display") or "").strip()
        if action.get("row") is not None or action.get("col") is not None:
            return f"{name.upper()}(row={action.get('row')}, col={action.get('col')})"
        return name
    return str(action or "").strip()


class NoImpactBook:
    """LRU-bounded state/action counts belonging to one agent/path/level."""

    def __init__(self, *, confirm_repeats: int = CONFIRM_REPEATS,
                 max_pairs: int = MAX_PAIRS_PER_PATH) -> None:
        self.confirm_repeats = max(1, int(confirm_repeats))
        self.max_pairs = max(1, int(max_pairs))
        self._counts: OrderedDict[tuple[str, str], int] = OrderedDict()

    def reset(self) -> None:
        self._counts.clear()

    def observe(self, state_key: str, action: str, kind: str) -> bool:
        pair = (state_key, action)
        if kind == "real_change":
            self._counts.pop(pair, None)
            return False
        if not action or kind not in {"identical", "hud_only"}:
            return False
        previous = self._counts.pop(pair, 0)
        self._counts[pair] = min(self.confirm_repeats, previous + 1)
        while len(self._counts) > self.max_pairs:
            self._counts.popitem(last=False)
        return previous < self.confirm_repeats <= self._counts[pair]

    def is_no_impact(self, state_key: str, action: str) -> bool:
        return self._counts.get((state_key, action), 0) >= self.confirm_repeats

    def notes_for(self, state_key: str) -> list[str]:
        actions = sorted(action for (state, action), count in self._counts.items()
                         if state == state_key and count >= self.confirm_repeats)
        return [f"{action} is confirmed no-impact in this semantic state "
                "(HUD/timer-only or identical interior); try a different action."
                for action in actions[:MAX_NOTES_PER_STATE]]


def format_no_impact_note(notes: list[str]) -> str:
    return "No-impact memory: " + " ".join(notes) if notes else ""


def load_snapshot(state_path: Any) -> dict[str, Any] | None:
    """Read the documented runtime JSON, including externally executed RESETs."""
    payload = json.loads(Path(state_path).read_text(encoding="utf-8"))
    frame = payload.get("current_frame") or {}
    grid = parse_grid(frame.get("grid"))
    if grid is None:
        return None
    history = payload.get("history") or []
    last = history[-1] if history else {}
    # Both number and last position distinguish repeated RESETs, even if the
    # reset produces an identical frame or resets the engine's action counter.
    resets = [(i, (entry.get("frame") or {}).get("step"))
              for i, entry in enumerate(history)
              if action_key(entry.get("action")).upper() == "RESET"]
    return {
        "grid": grid, "key": semantic_key(grid),
        "step": int(frame["step"]), "level": int(frame["level"]),
        "reset_marker": (len(resets), resets[-1] if resets else None),
        "last_action": action_key(last.get("action")),
        "last_action_step": (last.get("frame") or {}).get("step"),
    }


class ScopedMemory:
    def __init__(self, *, max_paths: int = MAX_PATHS_PER_AGENT,
                 max_pairs: int = MAX_PAIRS_PER_PATH) -> None:
        self.max_paths = max(1, int(max_paths))
        self.max_pairs = max_pairs
        self.paths: OrderedDict[str, dict[str, Any]] = OrderedDict()
        self.note = ""
        self.in_tool = False

    def refresh(self, state_path: Any, *, loader: Callable[[Any], Any] = load_snapshot,
                force: bool = False) -> tuple[dict[str, Any] | None, dict[str, Any] | None]:
        """Reuse only unchanged file snapshots; retain no raw JSON or history.

        The runtime writer atomically replaces the file, so inode/device detect
        replacements even when size and modification time are unchanged. Check
        metadata on both sides of a read, and never cache a racing read. Custom
        loaders retain their original behavior and do not require a real file.
        """
        path = str(Path(state_path).resolve())
        try:
            if loader is not load_snapshot:
                snapshot = loader(state_path)
                scope = self.sync(state_path, snapshot)
                if scope is not None:
                    scope.pop("cache_signature", None)
                return snapshot, scope

            def signature() -> tuple[int, ...]:
                stat = Path(path).stat()
                return (stat.st_dev, stat.st_ino, stat.st_size, stat.st_mtime_ns, stat.st_ctime_ns)

            before = signature()
            cached = self.paths.get(path)
            if not force and cached is not None and cached.get("cache_signature") == before:
                snapshot = cached["snapshot"]
                return snapshot, self.sync(state_path, snapshot)
            snapshot = loader(state_path)
            after = signature()
            if before != after:
                # A replacement during the read gets one fresh retry. If the
                # file remains unstable, leave advice empty until a later call.
                snapshot = loader(state_path)
                if after != signature():
                    raise OSError("S10 runtime state changed during snapshot read")
            scope = self.sync(state_path, snapshot)
            if scope is not None:
                scope["cache_signature"] = after
            return snapshot, scope
        except Exception:
            self.paths.pop(path, None)
            self.note = ""
            raise

    def sync(self, state_path: Any, snapshot: dict[str, Any] | None) -> dict[str, Any] | None:
        self.note = ""
        path = str(Path(state_path).resolve())
        if snapshot is None:
            self.paths.pop(path, None)
            return None
        scope = self.paths.pop(path, None)
        if scope is None:
            scope = {"book": NoImpactBook(max_pairs=self.max_pairs), "snapshot": None}
        previous = scope["snapshot"]
        if snapshot is not previous:
            scope.pop("cache_signature", None)
        if previous is not None and (
            snapshot["level"] != previous["level"]
            or snapshot["step"] < previous["step"]
            or snapshot["reset_marker"] != previous["reset_marker"]
        ):
            scope["book"].reset()
            _count("scope_resets")
        scope["snapshot"] = snapshot
        self.paths[path] = scope
        while len(self.paths) > self.max_paths:
            self.paths.popitem(last=False)
        self.note = format_no_impact_note(scope["book"].notes_for(snapshot["key"]))
        return scope


def memory_for(agent: Any) -> ScopedMemory:
    memory = getattr(agent, _S10_AGENT_ATTRIBUTE, None)
    if memory is None:
        memory = ScopedMemory()
        setattr(agent, _S10_AGENT_ATTRIBUTE, memory)
    return memory


def fresh_single_action(result: Any, previous_summary: Any, summary: Any,
                        before: dict[str, Any], after: dict[str, Any]) -> str | None:
    """Reject stale summaries, cancelling batches, and ambiguous attribution."""
    if not getattr(result, "step_executed", False):
        _count("skipped_no_action")
        return None
    if not isinstance(summary, dict) or summary is previous_summary:
        _count("skipped_unverified")
        return None
    if summary.get("executed_count") != 1:
        _count("skipped_multiple_actions")
        return None
    actions = summary.get("executed_actions")
    action = action_key(actions[0]) if isinstance(actions, list) and len(actions) == 1 else ""
    if not action or (
        after["step"] != before["step"] + 1
        or summary.get("start_action_num") != after["step"]
        or summary.get("end_action_num") != after["step"]
        or summary.get("level") != after["level"]
        or after["last_action_step"] != after["step"]
        or after["last_action"] != action
    ):
        _count("skipped_unverified")
        return None
    return action


def wrap_run_python_tool(original: Callable[..., Any], *,
                         loader: Callable[[Any], Any] = load_snapshot) -> Callable[..., Any]:
    @functools.wraps(original)
    def wrapped(self: Any, state_path: Any, arguments: dict[str, Any]) -> Any:
        memory = None
        before = None
        try:
            memory = memory_for(self)
            before, _scope = memory.refresh(state_path, loader=loader)
            memory.in_tool = True
        except Exception:
            _count("internal_errors")
            if memory is not None:
                memory.note = ""
        previous_summary = getattr(self, "_last_step_summary", None)
        # Keep the solver call outside instrumentation exception handling: its
        # original exception and traceback must reach the harness unchanged.
        try:
            result = original(self, state_path, arguments)
        finally:
            if memory is not None:
                memory.in_tool = False
                memory.note = ""
        try:
            if memory is None:
                return result
            after, scope = memory.refresh(state_path, loader=loader,
                                          force=bool(getattr(result, "step_executed", False)))
            if before is None or after is None or scope is None:
                return result
            summary = getattr(self, "_last_step_summary", None)
            fresh_summary = summary if summary is not previous_summary and isinstance(summary, dict) else {}
            reset = before["reset_marker"] != after["reset_marker"]
            terminal = any(fresh_summary.get(k) for k in ("level_transition", "run_complete", "game_over"))
            if reset or terminal or before["level"] != after["level"]:
                scope["book"].reset()
                memory.note = ""
                return result
            action = fresh_single_action(result, previous_summary, summary, before, after)
            if action and action.upper() != "RESET":
                kind = classify_transition(before["grid"], after["grid"])
                _count("transitions")
                _count(kind)
                if scope["book"].observe(before["key"], action, kind):
                    _count("confirmed_no_impact")
            memory.note = format_no_impact_note(scope["book"].notes_for(after["key"]))
        except Exception:
            _count("internal_errors")
            if memory is not None:
                memory.note = ""
        return result
    setattr(wrapped, S10_PATCH_FLAG, True)
    return wrapped


def wrap_ensure_session(original: Callable[..., Any], *,
                        loader: Callable[[Any], Any] = load_snapshot) -> Callable[..., Any]:
    @functools.wraps(original)
    def wrapped(self: Any, state_path: Any) -> Any:
        result = original(self, state_path)
        memory = None
        try:
            memory = memory_for(self)
            if not memory.in_tool:
                memory.refresh(state_path, loader=loader)
        except Exception:
            _count("internal_errors")
            if memory is not None:
                memory.note = ""
        return result
    setattr(wrapped, S10_PATCH_FLAG, True)
    return wrapped


def wrap_knowledge_lines(original: Callable[..., Any]) -> Callable[..., Any]:
    @functools.wraps(original)
    def wrapped(self: Any) -> list[str]:
        lines = original(self)
        try:
            note = memory_for(self).note
            if not note:
                return lines
            rendered = list(lines)
            prefix = "- Recent findings: "
            for index, line in enumerate(rendered):
                if line.startswith(prefix):
                    rendered[index] = prefix + note + " " + line[len(prefix):]
                    return rendered
            if rendered:
                rendered.insert(len(rendered) - 1, prefix + note)
                return rendered
            return ["Working world model carried from earlier turns:", prefix + note,
                    "- Revise any item above immediately if `current_frame` or `history` contradicts it."]
        except Exception:
            _count("internal_errors")
            return lines
    setattr(wrapped, S10_PATCH_FLAG, True)
    return wrapped


def validate_hook_contract(agent_class: Any) -> None:
    for name, arity in (("_run_python_tool", 3), ("_ensure_session", 2),
                        ("_summarized_knowledge_lines", 1), ("_summarize_step_sequence", 2)):
        method = getattr(agent_class, name, None)
        if not callable(method):
            raise RuntimeError(f"S10 harness contract missing callable ToolAgent.{name}")
        try:
            inspect.signature(method).bind(*([None] * arity))
        except (TypeError, ValueError) as exc:
            raise RuntimeError(f"S10 incompatible ToolAgent.{name} signature") from exc
    if getattr(agent_class._run_python_tool, "_arc_s4_patched", False):
        raise RuntimeError("S10 replaces S4; do not install the shared S4 hook first.")


def write_telemetry(working_dir: Any, payload: dict[str, Any]) -> Path | None:
    path = Path(working_dir) / S10_TELEMETRY_NAME
    try:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + "\n", encoding="utf-8")
        return path
    except Exception as exc:
        print(f"S10 telemetry write failed: {exc!r}", flush=True)
        return None


def install_s10_hooks(bm: Any, target: Any | None = None, *, working_dir: Path,
                     bundle_dir: Path | None = None, extra: dict[str, Any] | None = None,
                     source_text: str | None = None) -> dict[str, Any]:
    from inference.agent import tool_agent as ta

    validate_hook_contract(ta.ToolAgent)
    if not callable(getattr(bm, "run", None)):
        raise RuntimeError("S10 harness contract missing bm.run")
    if not getattr(ta.ToolAgent._run_python_tool, S10_PATCH_FLAG, False):
        ta.ToolAgent._ensure_session = wrap_ensure_session(ta.ToolAgent._ensure_session)
        ta.ToolAgent._summarized_knowledge_lines = wrap_knowledge_lines(ta.ToolAgent._summarized_knowledge_lines)
        ta.ToolAgent._run_python_tool = wrap_run_python_tool(ta.ToolAgent._run_python_tool)
    payload = {
        "experiment_id": (extra or {}).get("experiment_id", S10_EXPERIMENT_ID),
        "single_change": "Scoped S4 memory with fresh single-primitive attribution",
        "hud_top_rows": HUD_TOP_ROWS, "hud_bottom_rows": 0,
        "confirm_repeats": CONFIRM_REPEATS, "global_ban": False,
        "max_paths_per_agent": MAX_PATHS_PER_AGENT,
        "max_pairs_per_path": MAX_PAIRS_PER_PATH,
        "max_notes_per_state": MAX_NOTES_PER_STATE,
        "knowledge_overlay": True, "s4_replaced": True,
        "installed_at_epoch": time.time(), "working_dir": str(working_dir),
        "bundle_dir": str(bundle_dir) if bundle_dir is not None else None,
        "source_sha256": hashlib.sha256(source_text.encode()).hexdigest() if source_text else None,
        "target_max_runtime_s": getattr(target, "max_runtime_s", None),
        "extra": extra or {}, "counters": dict(_COUNTERS),
    }
    original_run = bm.run
    if not getattr(original_run, S10_PATCH_FLAG, False):
        @functools.wraps(original_run)
        async def wrapped_run(*args: Any, **kwargs: Any) -> Any:
            started = time.perf_counter()
            try:
                return await original_run(*args, **kwargs)
            finally:
                report = dict(payload)
                with _LOCK:
                    report["counters"] = dict(_COUNTERS)
                report["runtime_seconds"] = time.perf_counter() - started
                report["finished_at_epoch"] = time.time()
                write_telemetry(working_dir, report)
        setattr(wrapped_run, S10_PATCH_FLAG, True)
        bm.run = wrapped_run
    write_telemetry(working_dir, payload)
    print("S10 no-impact advice installed: per-agent/path, fresh single primitives, bounded memory", flush=True)
    return payload


### 3.4. Экспорт результатов и парное сравнение игр

In [ ]:
%%writefile /kaggle/working/arc3_evaluation.py
"""Strict, stdlib-only exports and matched-game comparisons of TAAF scores.

Scores come directly from GameRun.final_score; they are never re-scored or
rescaled. This module can also be embedded in a notebook cell without TAAF or
project imports. Export each pass separately: one record per expected game.
"""

from __future__ import annotations

import argparse
import json
import math
import random
import statistics
from pathlib import Path
from typing import Any

SCHEMA_VERSION = "arc3-sprint-run-v1"
COMPARISON_SCHEMA_VERSION = "arc3-sprint-paired-v1"
SCORE_SCALE = "framework-native"
SCORE_SOURCE = "taaf.GameRun.final_score"
FINAL_STATES = frozenset({"won", "gave_up", "cancelled"})
COMPARABLE_MODES = frozenset({"panel", "public25"})


def _number(value: Any, label: str, *, nonnegative: bool = False) -> float:
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise ValueError(f"{label} must be a finite number")
    result = float(value)
    if not math.isfinite(result) or (nonnegative and result < 0):
        raise ValueError(f"{label} must be finite" + (" and nonnegative" if nonnegative else ""))
    return result


def _integer(value: Any, label: str, *, minimum: int = 0) -> int:
    if isinstance(value, bool) or not isinstance(value, int) or value < minimum:
        raise ValueError(f"{label} must be an integer >= {minimum}")
    return value


def _ids(values: Any, label: str) -> list[str]:
    if not isinstance(values, (list, tuple)) or not values:
        raise ValueError(f"{label} must be a nonempty list of game IDs")
    if any(not isinstance(item, str) or not item.strip() for item in values):
        raise ValueError(f"{label} contains an invalid game ID")
    if len(set(values)) != len(values):
        raise ValueError(f"{label} contains duplicate game IDs")
    return list(values)


def _json_copy(value: Any) -> Any:
    try:
        return json.loads(json.dumps(value, allow_nan=False, sort_keys=True))
    except (TypeError, ValueError) as exc:
        raise ValueError(f"Config must contain finite JSON values: {exc}") from exc


def _records_by_id(summary: dict[str, Any]) -> dict[str, dict[str, Any]]:
    expected = _ids(summary.get("expected_game_ids"), "expected_game_ids")
    records = summary.get("games")
    if not isinstance(records, list):
        raise ValueError("games must be a list")
    result = {}
    for record in records:
        if not isinstance(record, dict):
            raise ValueError("Each game record must be an object")
        game_id = record.get("game_id")
        if not isinstance(game_id, str) or not game_id.strip():
            raise ValueError("Missing game_id")
        if game_id in result:
            raise ValueError(f"Duplicate game record: {game_id}")
        if record.get("state") not in FINAL_STATES:
            raise ValueError(f"{game_id}: invalid final state {record.get('state')!r}")
        _number(record.get("final_score"), f"{game_id}.final_score")
        total = _integer(record.get("total_levels"), f"{game_id}.total_levels", minimum=1)
        completed = _integer(record.get("levels_completed"), f"{game_id}.levels_completed")
        if completed > total:
            raise ValueError(f"{game_id}: levels_completed exceeds total_levels")
        if record.get("state") == "won" and completed != total:
            raise ValueError(f"{game_id}: won state requires all levels completed")
        _integer(record.get("action_count"), f"{game_id}.action_count")
        _number(record.get("runtime_s"), f"{game_id}.runtime_s", nonnegative=True)
        if not isinstance(record.get("baselines_available"), bool):
            raise ValueError(f"{game_id}: baselines_available must be a boolean")
        result[game_id] = record
    missing = sorted(set(expected) - result.keys())
    extra = sorted(result.keys() - set(expected))
    if missing or extra:
        raise ValueError(f"Game coverage mismatch: missing={missing}, unexpected={extra}")
    return result


def build_run_summary(
    bm: Any,
    profile: str,
    mode: str,
    expected_game_ids: list[str],
    elapsed_s: float,
    config: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Export one ended TAAF Benchmark pass, or raise ValueError on invalid data.

    Cancelled runs with framework-finalized scores remain in the sample.
    Hidden-baseline and smoke results may be exported but cannot be compared as
    quality evidence. Config should contain model, seed, budgets, and all other
    shared evaluation controls; profile identifies the deliberate treatment.
    """
    expected = _ids(expected_game_ids, "expected_game_ids")
    if not isinstance(profile, str) or not profile.strip():
        raise ValueError("profile must be a nonempty string")
    if not isinstance(mode, str) or not mode.strip():
        raise ValueError("mode must be a nonempty string")
    if getattr(bm, "n_passes", None) != 1:
        raise ValueError("Export exactly one pass per summary (n_passes must equal 1)")
    if getattr(bm, "end_time", None) is None:
        raise ValueError("Benchmark has not ended")
    if config is not None and not isinstance(config, dict):
        raise ValueError("config must be a dict or None")
    games = []
    for run in bm.game_runs:
        actions = list(run.actions_per_level)
        if len(actions) != run.number_of_levels:
            raise ValueError(f"{run.game_id}: actions_per_level length mismatch")
        for count in actions:
            _integer(count, f"{run.game_id}.actions_per_level")
        action_count = len(run.history)
        if sum(actions) != action_count:
            raise ValueError(f"{run.game_id}: action history and level counters disagree")
        games.append({
            "game_id": run.game_id,
            "state": run.state,
            "final_score": _number(run.final_score, f"{run.game_id}.final_score"),
            "levels_completed": run.levels_completed,
            "total_levels": run.number_of_levels,
            "action_count": action_count,
            "runtime_s": _number(run.final_wallclock_seconds, f"{run.game_id}.runtime_s", nonnegative=True),
            "baselines_available": run.base_actions_per_level is not None,
        })
    summary = {
        "schema_version": SCHEMA_VERSION,
        "profile": profile,
        "mode": mode,
        "score_source": SCORE_SOURCE,
        "score_scale": SCORE_SCALE,
        "expected_game_ids": expected,
        "n_passes": 1,
        "elapsed_s": _number(elapsed_s, "elapsed_s", nonnegative=True),
        "config": _json_copy(config or {}),
        "games": games,
    }
    indexed = _records_by_id(summary)
    summary["games"] = [indexed[game_id] for game_id in expected]
    summary["valid"] = True
    summary["mean_score"] = statistics.fmean(record["final_score"] for record in games)
    summary["quality_comparison_eligible"] = (
        mode in COMPARABLE_MODES and all(record["baselines_available"] for record in games)
    )
    summary["interpretation"] = (
        "Framework-native scores for these games only. Smoke and competition runs "
        "are operational checks, not quality evidence; hidden baselines make local "
        "scores unsuitable for quality comparisons."
    )
    return summary


def _write_json(path: str | Path, payload: dict[str, Any]) -> None:
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(destination)


def write_run_summary(
    bm: Any,
    output_path: str | Path,
    *,
    profile: str,
    mode: str,
    expected_game_ids: list[str],
    elapsed_s: float,
    config: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Validate, atomically write a UTF-8 JSON summary, and return its contents."""
    summary = build_run_summary(bm, profile, mode, expected_game_ids, elapsed_s, config)
    _write_json(output_path, summary)
    return summary


def _comparison_config(summary: dict[str, Any]) -> dict[str, Any]:
    config = summary.get("config")
    if not isinstance(config, dict):
        raise ValueError("Comparison requires a config object")
    if any(key not in config for key in ("model", "seed", "budgets")):
        raise ValueError("Comparison requires config.model, config.seed, and config.budgets")
    if not isinstance(config["model"], str) or not config["model"].strip():
        raise ValueError("Comparison requires a nonempty model reference")
    if config["seed"] is not None:
        _integer(config["seed"], "config.seed", minimum=-1)
    if not isinstance(config["budgets"], dict) or not config["budgets"]:
        raise ValueError("Comparison requires nonempty config.budgets")
    return _json_copy(config)


def _quantile(sorted_values: list[float], probability: float) -> float:
    position = (len(sorted_values) - 1) * probability
    lower = int(position)
    upper = min(lower + 1, len(sorted_values) - 1)
    return sorted_values[lower] + (position - lower) * (sorted_values[upper] - sorted_values[lower])


def compare_summaries(
    baseline: dict[str, Any],
    candidate: dict[str, Any],
    *,
    bootstrap_samples: int = 10000,
    bootstrap_seed: int = 0,
) -> dict[str, Any]:
    """Compare matched games under identical recorded evaluation controls.

    Bootstrap resamples game pairs, not levels or individual actions. The CI is
    descriptive of the selected sample and does not establish generalization.
    """
    _integer(bootstrap_samples, "bootstrap_samples", minimum=1)
    _integer(bootstrap_seed, "bootstrap_seed")
    indexed = []
    for label, summary in (("baseline", baseline), ("candidate", candidate)):
        if not isinstance(summary, dict) or summary.get("schema_version") != SCHEMA_VERSION:
            raise ValueError(f"{label}: unsupported summary schema")
        if summary.get("valid") is not True or summary.get("n_passes") != 1:
            raise ValueError(f"{label}: summary must be valid and single-pass")
        if summary.get("mode") not in COMPARABLE_MODES:
            raise ValueError(f"{label}: smoke/competition or unknown mode cannot be quality evidence")
        if summary.get("score_source") != SCORE_SOURCE or summary.get("score_scale") != SCORE_SCALE:
            raise ValueError(f"{label}: score source or scale mismatch")
        _number(summary.get("elapsed_s"), f"{label}.elapsed_s", nonnegative=True)
        records = _records_by_id(summary)
        if any(not record["baselines_available"] for record in records.values()):
            raise ValueError(f"{label}: hidden baselines cannot support a quality comparison")
        indexed.append(records)
    if baseline["mode"] != candidate["mode"]:
        raise ValueError("Evaluation mode mismatch")
    if indexed[0].keys() != indexed[1].keys():
        raise ValueError("Baseline and candidate game IDs differ")
    base_config, candidate_config = _comparison_config(baseline), _comparison_config(candidate)
    if base_config != candidate_config:
        changed = sorted(key for key in base_config.keys() | candidate_config.keys()
                         if key not in base_config or key not in candidate_config or base_config[key] != candidate_config[key])
        raise ValueError(f"Evaluation config mismatch: {changed}")
    per_game = []
    for game_id in sorted(indexed[0]):
        first, second = indexed[0][game_id], indexed[1][game_id]
        if first["total_levels"] != second["total_levels"]:
            raise ValueError(f"{game_id}: total_levels mismatch")
        per_game.append({
            "game_id": game_id,
            "baseline_score": first["final_score"],
            "candidate_score": second["final_score"],
            "delta": second["final_score"] - first["final_score"],
        })
    deltas = [record["delta"] for record in per_game]
    rng = random.Random(bootstrap_seed)
    means = sorted(statistics.fmean(rng.choices(deltas, k=len(deltas))) for _ in range(bootstrap_samples))
    stochastic = base_config["seed"] in (None, -1)
    warnings = [
        "Descriptive matched-game comparison; no claim of statistical significance or hidden-test generalization.",
        "The bootstrap assumes sampled games are independent; related/public games may violate this assumption.",
        "One run per profile does not estimate within-game run-to-run variability.",
    ]
    if len(deltas) < 30:
        warnings.append("Small selected sample: the interval may be unstable or degenerate; repeat across seeds and broader panels.")
    if stochastic:
        warnings.append("Stochastic unpaired sampling: game IDs are matched, but seed=None/-1 does not pair model RNG draws.")
    else:
        warnings.append("Equal recorded seeds do not by themselves guarantee identical model RNG draws or deterministic serving.")
    return {
        "schema_version": COMPARISON_SCHEMA_VERSION,
        "baseline_profile": baseline.get("profile"),
        "candidate_profile": candidate.get("profile"),
        "mode": baseline["mode"],
        "score_scale": SCORE_SCALE,
        "config": base_config,
        "sample_size": len(deltas),
        "pairing": "game_id; stochastic unpaired sampling" if stochastic else "game_id; equal recorded seed",
        "baseline_mean": statistics.fmean(record["baseline_score"] for record in per_game),
        "candidate_mean": statistics.fmean(record["candidate_score"] for record in per_game),
        "mean_delta": statistics.fmean(deltas),
        "wins": sum(delta > 0 for delta in deltas),
        "ties": sum(delta == 0 for delta in deltas),
        "losses": sum(delta < 0 for delta in deltas),
        "paired_bootstrap_ci": {
            "confidence": 0.95,
            "lower": _quantile(means, 0.025),
            "upper": _quantile(means, 0.975),
            "samples": bootstrap_samples,
            "seed": bootstrap_seed,
            "resampling_unit": "game pair",
            "method": "percentile with linear quantile interpolation",
        },
        "games": per_game,
        "warnings": warnings,
    }


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("baseline", type=Path)
    parser.add_argument("candidate", type=Path)
    parser.add_argument("--output", type=Path)
    parser.add_argument("--bootstrap-samples", type=int, default=10000)
    parser.add_argument("--bootstrap-seed", type=int, default=0)
    args = parser.parse_args(argv)
    try:
        baseline = json.loads(args.baseline.read_text(encoding="utf-8-sig"))
        candidate = json.loads(args.candidate.read_text(encoding="utf-8-sig"))
        comparison = compare_summaries(baseline, candidate, bootstrap_samples=args.bootstrap_samples,
                                       bootstrap_seed=args.bootstrap_seed)
        if args.output:
            _write_json(args.output, comparison)
        print(json.dumps(comparison, indent=2, allow_nan=False))
    except (OSError, ValueError, TypeError, KeyError) as exc:
        parser.error(str(exc))
    return 0


# Notebook cells have __name__ == '__main__' but no __file__.
if __name__ == "__main__" and "__file__" in globals():
    raise SystemExit(main())


## 4. Установка одного профиля и паспорт запуска

В рабочем процессе ставится ровно один memory hook. Смена профиля в уже
выполненной сессии запрещена: нужен Restart Session, чтобы не сложить патчи.
Файл `run_manifest.json` фиксирует настройки и SHA-256 фактических исходников
bundle. Dataset slugs могут обновляться; одинаковые имена не гарантируют одинаковый код.

| Режим | Игры | Секунд/игру | Действий/игру | Параллельность |
|---|---|---:|---:|---:|
| smoke | первые 2 публичные | 180 | 4 | 2 |
| panel | выбранные 8 или 9 | 600 по умолчанию | 80 | 4 |
| public25 | все 25 публичных | 7920 | без искусственного лимита | 28 |
| competition | все из gateway | 7920 | без искусственного лимита | 28 |

Seed влияет только на offline. В Phase B сохранена исходная стохастика S4.
Одинаковый seed не гарантирует битовую воспроизводимость GPU/vLLM.

In [ ]:
if str(WORKING_DIR) not in sys.path:
    sys.path.insert(0, str(WORKING_DIR))
from arc3_runtime import resolve_run_config, apply_run_config, source_fingerprints, write_manifest
from arc3_evaluation import write_run_summary
from inference.agent import tool_agent as ta

if globals().get("_ARC3_PROFILE_INSTALLED"):
    raise RuntimeError("Restart Session before reinstalling or changing the profile")
RUN_CONFIG = resolve_run_config(
    true_submission=TRUE_SUBMISSION, profile=PROFILE, offline_mode=OFFLINE_MODE,
    panel=EVAL_PANEL, eval_seed=EVAL_SEED, panel_seconds=PANEL_SECONDS,
    panel_actions=PANEL_ACTIONS, panel_concurrency=PANEL_CONCURRENCY,
)
apply_run_config(bm, target, RUN_CONFIG)
# This module constant is what the actual payload builder reads.
ta._LOCAL_ANALYZER_SEED = RUN_CONFIG["seed"]
hook_module = importlib.import_module(
    "arc3_s10_memory" if PROFILE == "s10_scoped" else "arc3_s4_baseline"
)
install = getattr(hook_module, "install_s10_hooks" if PROFILE == "s10_scoped" else "install_s4_hooks")
install(bm, target=target, working_dir=WORKING_DIR, bundle_dir=BUNDLE_DIR,
        extra={"profile": PROFILE, "mode": RUN_CONFIG["mode"], "parent_public_scores": [3.70, 3.38]},
        source_text=Path(hook_module.__file__).read_text(encoding="utf-8"))
_ARC3_PROFILE_INSTALLED = PROFILE

from inference.agent import prompts as prompt_module
INPUT_HASHES = source_fingerprints({
    "tool_agent": Path(ta.__file__), "prompts": Path(prompt_module.__file__),
    "serving_setup": BUNDLE_DIR / "serving_setup.py",
    "benchmark_initial": BUNDLE_DIR / "benchmark_initial.pkl",
    "deploy_target": BUNDLE_DIR / "deploy_target.pkl",
    "setup_commands": BUNDLE_DIR / "setup_commands.json",
})
EVALUATION_CONFIG = {
    "model": "keithtyser/qwen3-8-flash-next-nvfp4/PyTorch/radixark-modelopt-fp4/1", "seed": RUN_CONFIG["seed"],
    "budgets": RUN_CONFIG["budgets"], "panel": RUN_CONFIG["panel"],
    "serving_profile": PUBLIC25_VLLM_PROFILE_ENV, "input_hashes": INPUT_HASHES,
}
write_manifest(WORKING_DIR / "run_manifest.json", {
    "schema_version": 1, "profile": PROFILE, "run_config": RUN_CONFIG,
    "evaluation_config": EVALUATION_CONFIG, "true_submission": TRUE_SUBMISSION,
    "candidate_source_sha256": hashlib.sha256(Path(hook_module.__file__).read_bytes()).hexdigest(),
    "parent_notebook_sha256": "573f82fdf1e382afcb0d4952fb060385213042e1e192bd2c376d64b4d49e112f",
    "historical_lb": {"S4": [3.70, 3.38], "S10": None},
})
print(json.dumps(RUN_CONFIG, indent=2))


## 5. Получение игр и исполнение агента

Hidden run получает список сред только от gateway. Offline-панели не могут
ограничить скрытую оценку. Каждый проход создаёт игры один раз; RESET остаётся
действием текущей игры, а не способом обнулить расходы.

Публичные панели зафиксированы заранее на уровне целых игр. `held_public` —
условно отложенная часть известных публичных данных, не приватная выборка.

In [ ]:
# Build the live competition game list from the gateway's available environments.
def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# Build the offline game list from the competition's bundled environment files.
def _offline_games(env_dir: str):
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# The gateway can take a while to come up; poll until it answers.
def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


# Skip pre-run display and git-status copies; the staged identity pins the run.

# arc_agi reads RECORDINGS_DIR and ARC_API_KEY from env (ArcadeSpec carries neither); operation
# mode, environments dir, and base url are all passed explicitly via the spec, so no env is needed.
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

PUBLIC_GAME_IDS = tuple([
    "tn36-ef4dde99",
    "lf52-271a04aa",
    "cn04-2fe56bfb",
    "bp35-0a0ad940",
    "wa30-ee6fef47",
    "lp85-305b61c3",
    "r11l-495a7899",
    "tu93-0768757b",
    "sp80-589a99af",
    "m0r0-492f87ba",
    "vc33-5430563c",
    "ar25-0c556536",
    "ka59-38d34dbb",
    "sc25-635fd71a",
    "sk48-d8078629",
    "dc22-fdcac232",
    "cd82-fb555c5d",
    "ft09-0d8bbf25",
    "g50t-5849a774",
    "ls20-9607627b",
    "re86-8af5384d",
    "s5i5-18d95033",
    "sb26-7fbdac44",
    "su15-1944f8ab",
    "tr87-cd924810"
])


In [ ]:
EVAL_PANELS = {'schema_version': 1, 'created_at': '2026-09-07', 'source': 'offline PUBLIC_GAME_IDS from champion Flash v3 packaging', 'note': 'These are known public games, not a hidden holdout. All levels of a game stay in the same split.', 'seeds': [101, 202, 303], 'development': ['tn36-ef4dde99', 'lf52-271a04aa', 'cn04-2fe56bfb', 'bp35-0a0ad940', 'wa30-ee6fef47', 'lp85-305b61c3', 'r11l-495a7899', 'tu93-0768757b'], 'validation': ['sp80-589a99af', 'm0r0-492f87ba', 'vc33-5430563c', 'ar25-0c556536', 'ka59-38d34dbb', 'sc25-635fd71a', 'sk48-d8078629', 'dc22-fdcac232'], 'held_public': ['cd82-fb555c5d', 'ft09-0d8bbf25', 'g50t-5849a774', 'ls20-9607627b', 're86-8af5384d', 's5i5-18d95033', 'sb26-7fbdac44', 'su15-1944f8ab', 'tr87-cd924810']}


In [ ]:
from arc3_runtime import select_offline_ids

if TRUE_SUBMISSION:
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = _competition_games()
else:
    offline_games = _offline_games(str(COMPETITION_ROOT / "environment_files"))
    offline_by_id = {game.env_name: game for game in offline_games}
    if len(offline_by_id) != len(offline_games):
        raise RuntimeError("Duplicate offline game IDs")
    if set(offline_by_id) != set(PUBLIC_GAME_IDS):
        raise RuntimeError("Public input changed; review the pinned game list before evaluation")
    selected_ids = select_offline_ids(list(PUBLIC_GAME_IDS), EVAL_PANELS, RUN_CONFIG)
    bm.games = [offline_by_id[game_id] for game_id in selected_ids]

EXPECTED_GAME_IDS = [game.env_name for game in bm.games]
if not EXPECTED_GAME_IDS or len(set(EXPECTED_GAME_IDS)) != len(EXPECTED_GAME_IDS):
    raise RuntimeError("Empty or duplicate game selection")
print({"mode": RUN_CONFIG["mode"], "games": len(EXPECTED_GAME_IDS)})

budget = float(target.max_runtime_s)
soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(seconds=budget - 600.0)
if datetime.now() >= soft_end:
    raise RuntimeError("Setup exhausted the notebook budget")


### 5.1. Запуск и завершение

Watchdog исходного решения восстанавливает локальный vLLM при сбое. Общее мягкое
завершение оставляет 10 минут на сохранение файлов и остановку процессов.
`run_summary.json` содержит все выбранные игры, включая нулевые результаты;
ошибка или пропуск игры не превращаются в высокий средний балл.

In [ ]:
if str(BUNDLE_DIR) not in sys.path:
    sys.path.insert(0, str(BUNDLE_DIR))
import vllm_server_watchdog as vllm_watchdog

vllm_watchdog_setup = vllm_watchdog.load_setup(BUNDLE_DIR / "serving_setup.py")
vllm_watchdog.start_background(
    vllm_watchdog_setup,
    vllm_watchdog.WatchdogConfig(interval_seconds=15.0, request_timeout_seconds=5,
                                failure_threshold=4, max_restart_attempts=2),
)
run_started = time.monotonic()
try:
    await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=True)
    bm._save_json()
    try:
        summary = write_run_summary(
            bm, WORKING_DIR / "run_summary.json", profile=PROFILE, mode=RUN_CONFIG["mode"],
            expected_game_ids=EXPECTED_GAME_IDS, elapsed_s=time.monotonic() - run_started,
            config=EVALUATION_CONFIG,
        )
    except Exception as export_error:
        if not TRUE_SUBMISSION:
            raise
        # A diagnostic failure must not discard successfully played hidden games.
        # TAAF can return partial results after an individual game crashes.
        summary = {"valid": False, "mode": "competition", "games": [],
                   "diagnostic_error": repr(export_error),
                   "interpretation": "Operational export failed; official scoring is performed by Kaggle."}
        print(summary)
        try:
            write_manifest(WORKING_DIR / "run_summary.json", summary)
        except OSError as write_error:
            print(f"Optional diagnostic write failed: {write_error!r}")
    if not TRUE_SUBMISSION:
        if PROFILE == "s10_scoped":
            telemetry = json.loads((WORKING_DIR / "s10_telemetry.json").read_text())
            if telemetry["counters"]["internal_errors"]:
                raise RuntimeError("S10 instrumentation errors: inspect s10_telemetry.json")
        total_actions = sum(len(run.history) for run in bm.game_runs)
        if total_actions <= 0:
            raise RuntimeError("Model produced no actions; empty smoke cannot pass")
        if RUN_CONFIG["mode"] == "public25":
            from inference.tools.eval import evaluate_runs, save_score_file
            frozen = evaluate_runs([WORKING_DIR])
            save_score_file(frozen, run_dirs=[WORKING_DIR], output_path=WORKING_DIR / "score.json")
        # Platform placeholder for Phase A; this is NOT an evaluation result.
        # Phase B plays the actual gateway and never takes this branch.
        import pandas as pd
        pd.DataFrame([["1_0", "1", True, 1]],
                     columns=["row_id", "game_id", "end_of_game", "score"]).to_parquet(
                         WORKING_DIR / "submission.parquet", index=False)
        print(f"OFFLINE_AUDIT mode={RUN_CONFIG['mode']} games={len(bm.game_runs)} actions={total_actions}")
finally:
    try:
        vllm_watchdog.stop_background(timeout_seconds=15.0)
    finally:
        for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
            try:
                subprocess.run(command, shell=True, check=False, cwd=WORKING_DIR,
                               env=_command_env(), timeout=30.0)
            except subprocess.TimeoutExpired:
                print("Teardown exceeded 30 seconds; notebook exit will release its process")


## 6. Как читать результаты

- `run_manifest.json`: точные параметры, происхождение и хеши входного кода.
- `run_summary.json`: игры, пройденные уровни, действия, время и native score TAAF.
- `s10_telemetry.json`: события памяти кандидата; для S4 — `s4_telemetry.json`.
- `benchmark.json`, артефакты Duck: исходные данные диагностики.
- `score.json`: официальный exporter исходного harness, только для `public25`.
- `submission.parquet`: в offline технический placeholder, не измеренный LB.

RHAE поощряет прохождение уровней при малом расходе действий. Короткий smoke
может завершиться нулём без ошибки: 4 действий обычно недостаточно для решения.
Нулевой smoke нельзя использовать для выбора лучшего агента.

In [ ]:
from IPython.display import display
import pandas as pd
report = summary
print({key: value for key, value in report.items() if key not in {"games", "config"}})
display(pd.DataFrame(report["games"]))
if RUN_CONFIG["mode"] == "smoke":
    print("SMOKE ONLY: подтверждает исполнение, не качество и не Public LB.")


## 7. Следующие эксперименты без потери воспроизводимости

1. Сохраните этот кандидат и контроль S4 раздельно. На одной панели сначала
   `PROFILE="s4_baseline"`, затем `"s10_scoped"`; одинаковые модель, inputs, seed,
   бюджет и concurrency. Каждый профиль запускайте в новой сессии.
2. Используйте `OFFLINE_MODE="panel"`, сначала development, затем validation.
   Для устойчивости повторите несколько seed (например, 101/202/303). Seed задаёт
   контролируемый режим сравнения; он не переносится автоматически в production.
3. Скачайте `run_summary.json` из каждого запуска и сравните локально:

```bash
python -m src.sprint_evaluation baseline.json candidate.json --output comparison.json
```

   В отдельной среде можно сохранить ячейку `arc3_evaluation.py` и запустить её как
   обычный Python-скрипт. Сравнение проверяет набор игр/настройки, сохраняет нули,
   показывает дельты по играм и bootstrap-интервал. Публичные игры и малая выборка
   не доказывают качество на private; результаты не подбирать по одному удачному seed.
4. `held_public` проверяйте после выбора конфигурации; затем один `public25` при
   достаточном бюджете. Полный public25 дорогой и выключен по умолчанию.
5. Следующая алгоритмическая гипотеза: память проверенных причинных правил и
   короткое планирование с проверкой предсказанного результата. Отдельно проверить
   отказ от фиксированного HUD crop. Менять по одному механизму, контролируя не
   только score, но и глубину, число действий, crash rate и runtime.

**Почему есть шанс переноса:** один алгоритм для всех игр, факты только из
наблюдаемых переходов текущей среды, отсутствие таблицы решений по ID и ручного
подбора действий для hidden. Это принцип конструкции, не гарантия private score.

## Источники и авторство

Основа: [Tufa Labs Duck harness](https://www.kaggle.com/code/jeroencottaar/tufa-labs-duck-harness-june-30-milestone-winner)
и [Flash Next NVFP4 serving Keith Tyser](https://www.kaggle.com/code/keithtyser/duck-qwen3-8-flash-next-nvfp4-mtp).
Новые части проекта: S4/S10 memory hooks, конфигурация, отчётность и этот notebook.
Чужие компоненты сохраняют свои лицензии и авторство; notebook не выдаёт их за новую модель.

[Соревнование](https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-3),
[метрика](https://docs.arcprize.org/methodology),
[Competition Mode](https://docs.arcprize.org/toolkit/competition_mode),
[официальный Starter](https://github.com/arcprize/ARC-AGI-3-Kaggle-Starter).